# Lock-In Amplifier Driver Test Notebook

| Layer | Class | Key Methods |
|---|---|---|
| 1 | `Instrument` | `idn()` |
| 2 | `Scpi` | `reset()`, `clear()`, `error()`, `wait()`, `self_test()`, `operation_complete()`, `initialize()` |
| 3 | `Lockin` | Reference, Input, Filter, and Data acquisition methods |

**Instructions:** Run each cell sequentially. Verify the **Expected Result** after each cell. A known reference signal source is recommended.

---
## 0. Setup & Connection

In [ ]:
from piec.drivers.autodetect import autodetect

In [ ]:
# --- Option A: Auto-detect ---
lockin = autodetect("lockin", verbose=True)

# --- Option B: Manual ---
# from piec.drivers.lockin.srs830 import SRS830
# lockin = SRS830("GPIB0::8::INSTR")

✅ **PASS** if no error is raised.

---
## 1. Instrument & SCPI Tests

### 1.1 Identification (`idn`)

In [ ]:
idn_response = lockin.idn()
print("IDN Response:")
print(idn_response)

**Expected Result:** `Manufacturer, Model, Serial Number, Firmware Version`

✅ **PASS** if the string matches your physical instrument.

### 1.2 Reset (`reset`)

In [ ]:
lockin.reset()
print("Reset command sent.")

**Expected Result:** Lock-in returns to factory defaults.

✅ **PASS** if the display resets.

### 1.3 Clear, Error, Self Test, Wait, OPC, Initialize

In [ ]:
lockin.clear()
print("Clear:", "OK")

error_response = lockin.error()
print("Error:", error_response)

self_test_result = lockin.self_test()
print("Self-test:", self_test_result)

lockin.wait()
opc = lockin.operation_complete()
print("OPC:", opc)

lockin.initialize()
print("Initialize: OK")

**Expected Result:**
- Error: `0`
- Self-test: `0` (may take a few seconds)
- OPC: `1`

✅ **PASS** if all values match and no errors raised.

---
## 2. Reference Configuration Tests

### 2.1 Set Reference Source (`set_reference_source`)

In [ ]:
lockin.set_reference_source(reference_source='internal')
print("Reference source set to INTERNAL.")

**Expected Result:** The lock-in should use its **internal oscillator** as the reference. The display should indicate internal reference.

✅ **PASS** if reference source shows Internal.

### 2.2 Set Reference Frequency (`set_reference_frequency`)

In [ ]:
lockin.set_reference_frequency(frequency=1000)
print("Reference frequency set to 1 kHz.")

**Expected Result:** The reference frequency should read **1.000 kHz** on the display.

✅ **PASS** if frequency matches.

In [ ]:
lockin.set_reference_frequency(frequency=100)
print("Reference frequency set to 100 Hz.")

✅ **PASS** if frequency reads 100 Hz.

### 2.3 Set Reference Amplitude (`set_amplitude`)

In [ ]:
lockin.set_amplitude(amplitude=1.0)
print("Reference amplitude set to 1.0 V.")

**Expected Result:** The oscillator output amplitude should show **1.0 V** (RMS or Vpp depending on instrument).

✅ **PASS** if amplitude matches.

### 2.4 Set Harmonic (`set_harmonic`)

In [ ]:
lockin.set_harmonic(harmonic=1)
print("Harmonic set to 1 (fundamental).")

**Expected Result:** Lock-in detects at the **fundamental** (1st harmonic).

✅ **PASS** if harmonic indicator shows 1.

In [ ]:
lockin.set_harmonic(harmonic=2)
print("Harmonic set to 2 (second harmonic).")

**Expected Result:** Lock-in detects at the **2nd harmonic** (2× reference frequency).

✅ **PASS** if harmonic indicator shows 2.

In [ ]:
# Reset to fundamental
lockin.set_harmonic(harmonic=1)

### 2.5 Set Phase (`set_phase`)

In [ ]:
lockin.set_phase(phase=0.0)
print("Phase set to 0°.")

✅ **PASS** if phase reads 0°.

In [ ]:
lockin.set_phase(phase=90.0)
print("Phase set to 90°.")

**Expected Result:** Phase offset is **90°**. X and Y channels should swap roles.

✅ **PASS** if phase reads 90°.

In [ ]:
lockin.set_phase(phase=0.0)
print("Phase reset to 0°.")

### 2.6 Configure Reference — Combined (`configure_reference`)

In [ ]:
lockin.configure_reference(
    voltage=0.5,
    frequency=500,
    source='internal',
    phase=0.0,
    harmonic=1
)
print("Reference configured: 0.5 V, 500 Hz, internal, 0°, 1st harmonic.")

✅ **PASS** if all reference parameters match on the display.

---
## 3. Input Configuration Tests

### 3.1 Set Input Configuration (`set_input_configuration`)

In [ ]:
# Set input to single-ended (e.g., 'A' on SRS830)
lockin.set_input_configuration(configuration='A')
print("Input configuration set to A (single-ended).")

**Expected Result:** Input is set to **single-ended** (or channel A). The indicator should show the selected input config.

✅ **PASS** if input configuration matches.

### 3.2 Set Input Coupling (`set_input_coupling`)

In [ ]:
lockin.set_input_coupling(coupling='AC')
print("Input coupling set to AC.")

**Expected Result:** Input coupling is **AC** (DC component removed).

✅ **PASS** if coupling shows AC.

In [ ]:
lockin.set_input_coupling(coupling='DC')
print("Input coupling set to DC.")

✅ **PASS** if coupling shows DC.

### 3.3 Set Sensitivity (`set_sensitivity`)

In [ ]:
lockin.set_sensitivity(sensitivity=1e-3)
print("Sensitivity set to 1 mV.")

**Expected Result:** The sensitivity should read **1 mV** (or equivalent full-scale range).

✅ **PASS** if sensitivity matches.

### 3.4 Set Notch Filter (`set_notch_filter`)

In [ ]:
lockin.set_notch_filter(notch_filter=1)
print("Notch filter enabled.")

**Expected Result:** The line notch filter(s) should be engaged.

✅ **PASS** if notch filter is active on display.

### 3.5 Configure Input — Combined (`configure_input`)

In [ ]:
lockin.configure_input(
    input_configuration='A',
    input_coupling='AC',
    input_line_notch=0
)
print("Input configured: A, AC coupling, notch off.")

✅ **PASS** if all input parameters match.

---
## 4. Filter & Gain Configuration Tests

### 4.1 Set Time Constant (`set_time_constant`)

In [ ]:
lockin.set_time_constant(time_constant=0.1)
print("Time constant set to 100 ms.")

**Expected Result:** Time constant reads **100 ms**.

✅ **PASS** if time constant matches.

In [ ]:
lockin.set_time_constant(time_constant=1.0)
print("Time constant set to 1 s.")

✅ **PASS** if time constant reads 1 s.

### 4.2 Set Filter Slope (`set_filter_slope`)

In [ ]:
lockin.set_filter_slope(filter_slope=12)
print("Filter slope set to 12 dB/oct.")

**Expected Result:** Low-pass filter slope is **12 dB/oct**.

✅ **PASS** if filter slope matches.

### 4.3 Configure Gain & Filters — Combined (`configure_gain_filters`)

In [ ]:
lockin.configure_gain_filters(
    sensitivity=1e-3,
    time_constant=0.3,
    lp_filter_slope=24
)
print("Gain/Filters configured: 1 mV sensitivity, 300 ms TC, 24 dB/oct.")

✅ **PASS** if all gain/filter parameters match.

---
## 5. Auto Functions

### 5.1 Auto Gain (`auto_gain`)

In [ ]:
lockin.auto_gain()
print("Auto gain executed.")

**Expected Result:** The sensitivity auto-adjusts to fit the current signal level.

✅ **PASS** if sensitivity changes to an appropriate value.

### 5.2 Auto Phase (`auto_phase`)

In [ ]:
lockin.auto_phase()
print("Auto phase executed.")

**Expected Result:** The phase auto-adjusts so the signal appears primarily in the X channel (Y ≈ 0).

✅ **PASS** if Y reading drops near zero.

---
## 6. Data Acquisition Tests

### 6.1 Quick Read (`quick_read` / `get_X_Y`)

In [ ]:
xy = lockin.quick_read()
print(f"Quick Read (X, Y): {xy}")

**Expected Result:** A tuple `(X, Y)` of float values.

✅ **PASS** if two numeric values are returned.

In [ ]:
# Legacy alias
xy2 = lockin.get_X_Y()
print(f"get_X_Y: {xy2}")

✅ **PASS** if result matches `quick_read`.

### 6.2 Read Data (`read_data`)

In [ ]:
data = lockin.read_data()
print("Full Read Data:")
print(data)

**Expected Result:** A dict with keys `X`, `Y`, `R`, `Theta`.

✅ **PASS** if all four values are present and numeric.

### 6.3 Individual Channel Reads

In [ ]:
x = lockin.get_X()
y = lockin.get_Y()
r = lockin.get_R()
theta = lockin.get_theta()
print(f"X = {x}")
print(f"Y = {y}")
print(f"R = {r}")
print(f"Theta = {theta}")

**Expected Result:** Four individual float values.
- R should equal √(X² + Y²)
- Theta should equal arctan(Y/X)

✅ **PASS** if all four values are reasonable and mathematically consistent.

---
## 7. Cleanup

In [ ]:
lockin.reset()
print("Lock-in reset to factory defaults.")

---
## Test Summary

| # | Method(s) Tested | Pass/Fail |
|---|------------------|-----------|
| 0 | `__init__` / `autodetect` | |
| 1.1 | `idn()` | |
| 1.2 | `reset()` | |
| 1.3 | `clear()`, `error()`, `self_test()`, `wait()`, `operation_complete()`, `initialize()` | |
| 2.1 | `set_reference_source()` | |
| 2.2 | `set_reference_frequency()` | |
| 2.3 | `set_amplitude()` | |
| 2.4 | `set_harmonic()` | |
| 2.5 | `set_phase()` | |
| 2.6 | `configure_reference()` | |
| 3.1 | `set_input_configuration()` | |
| 3.2 | `set_input_coupling()` | |
| 3.3 | `set_sensitivity()` | |
| 3.4 | `set_notch_filter()` | |
| 3.5 | `configure_input()` | |
| 4.1 | `set_time_constant()` | |
| 4.2 | `set_filter_slope()` | |
| 4.3 | `configure_gain_filters()` | |
| 5.1 | `auto_gain()` | |
| 5.2 | `auto_phase()` | |
| 6.1 | `quick_read()` / `get_X_Y()` | |
| 6.2 | `read_data()` | |
| 6.3 | `get_X()`, `get_Y()`, `get_R()`, `get_theta()` | |
| 7 | `reset()` | |

**Technician Signature:** _______________  
**Date:** _______________  
**Instrument Serial #:** _______________